# Dataset Profiling

Profile the local fraud datasets before training. Reads are streamed or chunked because PaySim is larger than 490 MB.

In [ ]:
from collections import Counter
from pathlib import Path
import csv

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CREDIT_CARD_PATH = ROOT / 'creditcard.csv'
PAYSIM_PATH = ROOT / 'PS_20174392719_1491204439457_log.csv'
assert CREDIT_CARD_PATH.exists(), CREDIT_CARD_PATH
assert PAYSIM_PATH.exists(), PAYSIM_PATH

## Streaming schema and target profile

In [ ]:
def stream_profile(path: Path, target: str, time_column: str) -> dict:
    row_count = 0
    target_counts = Counter()
    first_time = None
    last_time = None
    with path.open('r', newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        columns = reader.fieldnames
        for row in reader:
            row_count += 1
            target_counts[row[target]] += 1
            first_time = row[time_column] if first_time is None else first_time
            last_time = row[time_column]
    return {
        'path': path.name,
        'rows': row_count,
        'columns': columns,
        'target_counts': dict(target_counts),
        'positive_rate': target_counts.get('1', 0) / row_count,
        'first_time': first_time,
        'last_time': last_time,
    }

credit_profile = stream_profile(CREDIT_CARD_PATH, 'Class', 'Time')
paysim_profile = stream_profile(PAYSIM_PATH, 'isFraud', 'step')
credit_profile, paysim_profile

In [ ]:
credit = pd.read_csv(CREDIT_CARD_PATH)
credit.info()
credit.groupby('Class')[['Amount', 'Time']].agg(['count', 'mean', 'median', 'max'])

sample_parts = []
for chunk in pd.read_csv(PAYSIM_PATH, chunksize=250_000):
    sample_parts.append(chunk.sample(n=min(5_000, len(chunk)), random_state=42))
    if len(sample_parts) == 12:
        break
paysim_sample = pd.concat(sample_parts, ignore_index=True)
paysim_sample.groupby('isFraud')[['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']].agg(['count', 'mean', 'median', 'max'])

## Leakage review

Do not pass `nameOrig` or `nameDest` directly to a baseline model. Keep `isFlaggedFraud` documented as an existing rule signal and compare models with and without it.

In [ ]:
identifier_columns = {'nameOrig', 'nameDest'}
candidate_numeric_columns = {'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'}
assert identifier_columns.isdisjoint(candidate_numeric_columns)
paysim_sample[list(candidate_numeric_columns) + ['isFraud']].corr(numeric_only=True)['isFraud'].sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
credit.groupby('Class')['Amount'].mean().plot.bar(ax=axes[0], title='Credit-card mean amount')
paysim_sample.groupby('isFraud')['amount'].mean().plot.bar(ax=axes[1], title='PaySim sampled mean amount')
plt.tight_layout()
plt.show()